# LoCI LAMP Product Creation - Setup Phase

This notebook sets up all the necessary components for creating the LoCI LAMP product with DPP.

**Flow mirrors the GUI CreateProjectForm.tsx:**
1. User authentication and registration
2. Location setup
3. Units and resource specifications
4. Process definitions
5. Initial resources and components creation
6. Image asset preparation

In [ ]:
# Module imports and auto-reload setup
%load_ext autoreload
%aimport if_lib, if_utils, if_dpp, if_graphics, if_consts, if_gc1dpp
%autoreload 1
import os
import json
import random
import csv
from pathlib import Path

from if_utils import get_filename, show_data, save_traces

from if_lib import generate_random_challenge, read_HMAC, read_keypair, get_id_person, get_location_id, \
get_unit_id, get_resource_spec_id, get_resource, get_process, create_event, make_transfer, reduce_resource, set_user_location

from if_gc1dpp import upload_file_on_dpp, calculate_file_checksum

## Configuration and Endpoints

In [ ]:
# Define constants for this use case
USE_CASE = 'locilamp'

# Zenflows API endpoint
ENDPOINT = 'https://proxy.dpp-staging.dnstest.dyne.org/zenflows/api'

# DPP service endpoint
DPP_URL = 'https://proxy.dpp-staging.dnstest.dyne.org/interfacer-dpp'

# CSV file with product data
CSV_FILE = Path('20260127 LOCI LAMP product informaiton DPP DE.csv')

# Assets directory
ASSETS_DIR = Path('assets')

# Participants for LoCI LAMP production
USERS = ['tchibo', 'locilamp_designer', 'locilamp_manufacturer']

print(f"Configuration:")
print(f"  Use Case: {USE_CASE}")
print(f"  Zenflows: {ENDPOINT}")
print(f"  DPP URL: {DPP_URL}")
print(f"  CSV File: {CSV_FILE}")
print(f"  Assets: {ASSETS_DIR}")

## File Path Configuration

In [ ]:
# Calculate names of settings files
USERS_FILE = get_filename('cred_users.json', ENDPOINT, USE_CASE)
LOCS_FILE = get_filename('loc_users.json', ENDPOINT, USE_CASE)
UNITS_FILE = get_filename('units_data.json', ENDPOINT, USE_CASE)
SPECS_FILE = get_filename('res_spec_data.json', ENDPOINT, USE_CASE)
DPP_FILE = get_filename('dpp_data.json', ENDPOINT, USE_CASE)
RES_FILE = get_filename('initial_resources.json', ENDPOINT, USE_CASE)
PROCESS_FILE = get_filename('process_data.json', ENDPOINT, USE_CASE)
IMAGES_FILE = get_filename('images_data.json', ENDPOINT, USE_CASE)

print(f"Data will be saved to:")
print(f"  Users: {USERS_FILE}")
print(f"  Locations: {LOCS_FILE}")
print(f"  Units: {UNITS_FILE}")
print(f"  Resource Specs: {SPECS_FILE}")
print(f"  DPP Data: {DPP_FILE}")
print(f"  Initial Resources: {RES_FILE}")
print(f"  Processes: {PROCESS_FILE}")
print(f"  Images: {IMAGES_FILE}")

## Load Product Data from CSV

Parse the LOCI LAMP product information from the CSV file.

In [ ]:
# Parse LOCI LAMP CSV file
def parse_loci_lamp_csv(csv_path):
    """Parse LOCI LAMP CSV file into structured product data."""
    product_data = {}
    current_category = None
    
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter=';')
        next(reader)  # Skip header
        for row in reader:
            if len(row) < 4:
                continue
            category = row[0].strip() if row[0].strip() else current_category
            if row[0].strip():
                current_category = category
            title = row[1].strip() if len(row) > 1 else ''
            value = row[3].strip() if len(row) > 3 else ''
            
            if title and value:
                if category not in product_data:
                    product_data[category] = {}
                product_data[category][title] = value
    
    return product_data

# Load the CSV data
print("Loading LOCI LAMP product data from CSV...")
loci_lamp_data = parse_loci_lamp_csv(CSV_FILE)

print(f"\n✓ Loaded {len(loci_lamp_data)} categories:")
for category in loci_lamp_data:
    print(f"  - {category}: {len(loci_lamp_data[category])} fields")

# Extract key product info
product_overview = loci_lamp_data.get('Product Overview', {})
print(f"\nProduct Information:")
print(f"  Brand: {product_overview.get('Brand Name', 'N/A')}")
print(f"  Name: {product_overview.get('Product Name', 'N/A')}")
print(f"  Model: {product_overview.get('Model Name', 'N/A')}")

## Initialize Data Structures

In [ ]:
# Create data structures
process_data = {}
res_data = {}
event_seq = []
dpp_data = {}
images_data = {}

# Initialize or load user data - LoCI LAMP specific users
if os.path.isfile(USERS_FILE):
    with open(USERS_FILE,'r') as f:
        users_data = json.loads(f.read())
    print("Credentials file available for users")
else:
    users_data = {}
    # Tchibo - the brand owner
    users_data['tchibo'] = {
      "userChallenges": {
        "whereParentsMet": "Hamburg",
        "nameFirstPet": "Kaffee",
        "nameFirstTeacher": "Hans",
        "whereHomeTown": "Hamburg",
        "nameMotherMaid": "Schmidt"
      },
      "name": "Tchibo GmbH",
      "username": "tchibo_locilamp",
      "email": "service@tchibo.de",
      "note": "Tchibo GmbH - LOCI LAMP brand"
    }
    # Designer of the lamp
    users_data['locilamp_designer'] = {
      "userChallenges": {
        "whereParentsMet": "Berlin",
        "nameFirstPet": "Licht",
        "nameFirstTeacher": "Maria",
        "whereHomeTown": "Munich",
        "nameMotherMaid": "Weber"
      },
      "name": "LoCI LAMP Designer",
      "username": "locilamp_designer",
      "email": "designer@locilamp.de",
      "note": "LOCI LAMP product designer"
    }
    # Manufacturer
    users_data['locilamp_manufacturer'] = {
      "userChallenges": {
        "whereParentsMet": "Frankfurt",
        "nameFirstPet": "Holz",
        "nameFirstTeacher": "Peter",
        "whereHomeTown": "Stuttgart",
        "nameMotherMaid": "Müller"
      },
      "name": "LoCI LAMP Manufacturer",
      "username": "locilamp_manufacturer",
      "email": "manufacture@locilamp.de",
      "note": "LOCI LAMP manufacturer"
    }
    with open(USERS_FILE,'w') as f:
        json.dump(users_data, f)
    print("Created new user credentials file")

# Initialize or load location data
if os.path.isfile(LOCS_FILE):
    with open(LOCS_FILE,'r') as f:
        locs_data = json.loads(f.read())
    print("Location file available")
else:
    locs_data = {}
    # Tchibo HQ in Hamburg
    locs_data['tchibo'] = {
        "name": "Tchibo GmbH Headquarters",
        "lat": 53.6040379,
        "long": 10.0221277,
        "addr": "Überseering 18, 22297 Hamburg, Germany",
        "note": "Tchibo headquarters"
    }
    # Designer location
    locs_data['locilamp_designer'] = {
        "name": "LoCI LAMP Design Studio",
        "lat": 52.5200066,
        "long": 13.404954,
        "addr": "Design Studio, Berlin, Germany",
        "note": "LOCI LAMP design location"
    }
    # Manufacturer location in Germany
    locs_data['locilamp_manufacturer'] = {
        "name": "LoCI LAMP Manufacturing",
        "lat": 48.7758459,
        "long": 9.1829321,
        "addr": "Manufacturing Facility, Stuttgart, Germany",
        "note": "LOCI LAMP manufacturing location"
    }
    with open(LOCS_FILE,'w') as f:
        json.dump(locs_data, f)
    print("Created new location file")

# Initialize units and specs
if os.path.isfile(UNITS_FILE):
    with open(UNITS_FILE,'r') as f:
        units_data = json.loads(f.read())
    print(f"Unit file available")
else:
    units_data = {}

if os.path.isfile(SPECS_FILE):
    with open(SPECS_FILE,'r') as f:
        res_spec_data = json.loads(f.read())
    print(f"Resource Spec file available")
else:
    res_spec_data = {}

## Authentication Setup: HMAC Generation

In [ ]:
# Read HMAC or get it from the server
for user in USERS:
    read_HMAC(USERS_FILE, users_data, user, endpoint=ENDPOINT)

## Cryptographic Key Generation

In [ ]:
# Read the keypair for each user
for user in USERS:
    read_keypair(USERS_FILE, users_data, user)

## User Registration in Zenflows

In [ ]:
# Read or get id of the person
for user in USERS:
    get_id_person(USERS_FILE, users_data, user, endpoint=ENDPOINT)

## Location Registration and Assignment

In [ ]:
# Read or get the location id and set user locations
for user in USERS:
    get_location_id(LOCS_FILE, users_data[user], locs_data, user, endpoint=ENDPOINT)
    set_user_location(USERS_FILE, users_data, locs_data, user, endpoint=ENDPOINT)

## Unit of Measurement Registration

In [ ]:
# Get the ids of all units
get_unit_id(UNITS_FILE, users_data['tchibo'], units_data, 'piece', 'u_piece', 'om2:one', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['tchibo'], units_data, 'mass', 'kg', 'om2:kilogram', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['tchibo'], units_data, 'time', 'h', 'om2:hour', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['tchibo'], units_data, 'energy', 'kWh', 'om2:kilowattHour', endpoint=ENDPOINT)

## Fetch GUI instanceVariables Specs

Fetch the standard resource specifications from the server that match the GUI's instanceVariables:
- `specProjectDesign` - for creating designs
- `specProjectProduct` - for creating products  
- `specMaterial` - for materials
- `specMachine` - for machines
- `specDpp` - for DPP resources

In [ ]:
# ============================================================================
# FETCH GUI instanceVariables SPECS FROM SERVER
# ============================================================================
# These are the standard specs used by the GUI for different resource types.
# We need to use these same specs to match the GUI flow exactly.

import requests

def fetch_instance_variables_specs(user_data, endpoint):
    """Fetch the instanceVariables specs from the Zenflows server."""
    query = """
    query {
        instanceVariables {
            specs {
                specProjectDesign { id name }
                specProjectProduct { id name }
                specProjectService { id name }
                specMachine { id name }
                specMaterial { id name }
                specCurrency { id name }
                specDpp { id name }
            }
            units {
                unitOne { id label symbol }
            }
        }
    }
    """
    
    headers = {
        'Content-Type': 'application/json',
        'zenflows-id': user_data['id'],
        'zenflows-sign': user_data['seedServerSideShard.HMAC']
    }
    
    response = requests.post(
        endpoint,
        json={'query': query},
        headers=headers
    )
    
    if response.status_code == 200:
        data = response.json()
        return data.get('data', {}).get('instanceVariables', {})
    else:
        print(f"Error fetching instanceVariables: {response.status_code}")
        print(response.text)
        return None

# Fetch the specs
instance_vars = fetch_instance_variables_specs(users_data['tchibo'], ENDPOINT)

if instance_vars:
    gui_specs = instance_vars.get('specs', {})
    gui_units = instance_vars.get('units', {})
    
    print("✅ GUI instanceVariables specs fetched:")
    print(f"   specProjectDesign: {gui_specs.get('specProjectDesign', {}).get('id', 'N/A')}")
    print(f"   specProjectProduct: {gui_specs.get('specProjectProduct', {}).get('id', 'N/A')}")
    print(f"   specMaterial: {gui_specs.get('specMaterial', {}).get('id', 'N/A')}")
    print(f"   specMachine: {gui_specs.get('specMachine', {}).get('id', 'N/A')}")
    print(f"   specDpp: {gui_specs.get('specDpp', {}).get('id', 'N/A')}")
    print(f"   unitOne: {gui_units.get('unitOne', {}).get('id', 'N/A')}")
    
    # Store these for use in resource creation
    SPEC_PROJECT_DESIGN_ID = gui_specs.get('specProjectDesign', {}).get('id')
    SPEC_PROJECT_PRODUCT_ID = gui_specs.get('specProjectProduct', {}).get('id')
    SPEC_MATERIAL_ID = gui_specs.get('specMaterial', {}).get('id')
    SPEC_MACHINE_ID = gui_specs.get('specMachine', {}).get('id')
    SPEC_DPP_ID = gui_specs.get('specDpp', {}).get('id')
    UNIT_ONE_ID = gui_units.get('unitOne', {}).get('id')
else:
    print("⚠️ Could not fetch instanceVariables, will use local specs")

## Process Definitions for LoCI LAMP Production

In [ ]:
# Create the processes for LoCI LAMP production

# Process for creating KROMA KRAFT cardboard components
process_name = 'Create_locilamp_cardboard_components'
user_data = users_data['locilamp_manufacturer']
note = f"Creation of KROMA KRAFT cardboard lamelles and base by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Process for creating the lampshade
process_name = 'Create_locilamp_lampshade'
user_data = users_data['locilamp_manufacturer']
note = f"Creation of transparent acid-free paper lampshade by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Process for assembling electrical components
process_name = 'Create_locilamp_electrical'
user_data = users_data['locilamp_manufacturer']
note = f"Assembly of textile cable, E27 socket, and switch by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Process for creating the LOCI LAMP design
process_name = 'Create_locilamp_design'
user_data = users_data['locilamp_designer']
note = f"Creation of LOCI LAMP V 2.0 design by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Process for final assembly of LOCI LAMP
process_name = 'Assemble_locilamp'
user_data = users_data['tchibo']
note = f"Final assembly and packaging of LOCI LAMP by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Save process data to file
with open(PROCESS_FILE, 'w') as f:
    json.dump(process_data, f, indent=2)
print(f"Process data saved to {PROCESS_FILE}")

## Resource Specification Registration

In [ ]:
# Register all resource specifications for LOCI LAMP components

# Raw material: KROMA KRAFT Displaykarton (FSC-certified)
name = 'kroma_kraft_cardboard'
note = 'KROMA KRAFT Displaykarton - PVC-free, FSC-certified cardboard'
classification = 'https://www.wikidata.org/wiki/Q389782'  # Cardboard
default_unit_id = units_data['mass']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Raw material: Transparent acid-free paper (FSC)
name = 'acidfree_paper'
note = 'Transparent acid-free paper - FSC-certified'
classification = 'https://www.wikidata.org/wiki/Q11472'  # Paper
default_unit_id = units_data['mass']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Electrical components: Textile cable
name = 'textile_cable'
note = 'Textile cable H03VVH2-F 2×0.75 mm² - Polyester, PVC, Copper'
classification = 'https://www.wikidata.org/wiki/Q199647'  # Electrical cable
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Electrical components: E27 socket
name = 'e27_socket'
note = 'E27 lamp socket - Copper, PET, galvanized steel'
classification = 'https://www.wikidata.org/wiki/Q5406598'  # Light socket
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Electrical components: EU plug
name = 'eu_plug'
note = 'EU plug (2-PIN) - Copper and PVC'
classification = 'https://www.wikidata.org/wiki/Q190642'  # Electrical plug
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Electrical components: Toggle switch
name = 'toggle_switch'
note = 'Toggle switch - Copper and PP'
classification = 'https://www.wikidata.org/wiki/Q163607'  # Switch
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Semi-finished: Cardboard lamelles and base
name = 'locilamp_cardboard_components'
note = 'LOCI LAMP lamelles and core base from KROMA KRAFT cardboard'
classification = 'https://github.com/locilamp/cardboard-components'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Semi-finished: Lampshade
name = 'locilamp_lampshade'
note = 'LOCI LAMP transparent acid-free paper lampshade'
classification = 'https://github.com/locilamp/lampshade'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Semi-finished: Electrical assembly
name = 'locilamp_electrical_assembly'
note = 'LOCI LAMP electrical assembly - cable, socket, plug, switch'
classification = 'https://github.com/locilamp/electrical'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Design: LOCI LAMP design
name = 'locilamp_design'
note = 'LOCI LAMP V 2.0 design specification'
classification = 'https://github.com/locilamp/design'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_designer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Final product: LOCI LAMP
name = 'locilamp'
note = 'LOCI LAMP V 2.0 - Complete table lamp self-assembly kit'
classification = 'https://www.wikidata.org/wiki/Q1146001'  # Table lamp
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['tchibo'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# ============================================================================
# CRITICAL SPECS FOR PRODUCTION NOTEBOOK (matches GUI instanceVariables)
# ============================================================================

# specProjectProduct - for creating products (locilamp_assembled)
name = 'locilamp_assembled'
note = 'LOCI LAMP V 2.0 - Assembled product specification (specProjectProduct)'
classification = 'https://www.wikidata.org/wiki/Q1146001'  # Table lamp
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['tchibo'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# specDpp - for creating DPP resources (required by GUI CREATE_DPP_RESOURCE mutation)
name = 'specDpp'
note = 'Digital Product Passport resource specification'
classification = 'https://dpp.2050.cloud/spec/dpp'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['tchibo'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# ============================================================================
# Work specifications
# ============================================================================
name = 'design_work'
note = 'Specification for design work'
classification = 'https://www.wikidata.org/wiki/Q82604'
default_unit_id = units_data['time']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_designer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'manufacturing_work'
note = 'Specification for manufacturing work'
classification = 'https://www.wikidata.org/wiki/Q187939'
default_unit_id = units_data['time']['id']
get_resource_spec_id(SPECS_FILE, users_data['locilamp_manufacturer'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

print(f"✓ Registered {len(res_spec_data)} resource specifications")
print(f"  - locilamp_assembled (specProjectProduct): {res_spec_data.get('locilamp_assembled', {}).get('id', 'N/A')}")
print(f"  - specDpp: {res_spec_data.get('specDpp', {}).get('id', 'N/A')}")

## Initial Raw Materials Creation

In [ ]:
# Create the initial raw materials

# KROMA KRAFT cardboard (0.3 kg based on ~0.345 kg lamp weight)
res_name = 'kroma_kraft_cardboard'
amount = 0.3
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# Acid-free paper for lampshade
res_name = 'acidfree_paper'
amount = 0.02
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# Textile cable
res_name = 'textile_cable'
amount = 1
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# E27 socket
res_name = 'e27_socket'
amount = 1
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# EU plug
res_name = 'eu_plug'
amount = 1
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# Toggle switch
res_name = 'toggle_switch'
amount = 1
get_resource(res_data, res_spec_data, res_name, users_data['locilamp_manufacturer'], event_seq, amount, endpoint=ENDPOINT)

# Save initial resources to file
with open(RES_FILE, 'w') as f:
    json.dump(res_data, f, indent=2)
print(f"✓ Initial resources saved to {RES_FILE}")

## Create Material Resources (using GUI specMaterial)

Create material resources using the GUI's `specMaterial` specification.
These materials will be CONSUMED during production (matching MaterialsStep in the GUI).

In [ ]:
# ============================================================================
# CREATE MATERIAL RESOURCES (using GUI specMaterial)
# ============================================================================
# These materials use the GUI's specMaterial specification from instanceVariables.
# They will be CONSUMED during production, matching the MaterialsStep in the GUI.

from if_lib import send_signed
from datetime import datetime, timezone

material_resources = {}

# Helper function to create a material resource using specMaterial
def create_material_resource(name, note, user_data, spec_id, unit_id, amount=1, endpoint=ENDPOINT):
    """Create a material resource using the GUI's specMaterial.
    
    Uses the same mutation structure as GUI CREATE_PROJECT but for materials.
    Key: resourceConformsTo goes in the EVENT, not in newInventoriedResource.
    """
    
    # Check if resource already exists
    if name in material_resources and material_resources[name].get('id'):
        print(f"Material {name} already exists with ID: {material_resources[name]['id']}")
        return material_resources[name]
    
    # Use the library's send_signed for proper authentication
    ts = datetime.now(timezone.utc).isoformat()
    
    variables = {
        "event": {
            "action": "raise",
            "provider": user_data['id'],
            "receiver": user_data['id'],
            "hasPointInTime": ts,
            "resourceConformsTo": spec_id,
            "resourceQuantity": {
                "hasNumericalValue": amount,
                "hasUnit": unit_id
            },
            "toLocation": user_data.get('location_id')
        },
        "newInventoriedResource": {
            "name": name,
            "note": note
        }
    }
    
    query = """mutation($event:EconomicEventCreateParams!, $newInventoriedResource:EconomicResourceCreateParams) {
        createEconomicEvent(event:$event, newInventoriedResource:$newInventoriedResource) {
            economicEvent {
                id
                resourceInventoriedAs {
                    id
                    name
                }
            }
        }
    }"""
    
    result = send_signed(query, variables, user_data['username'], user_data['keyring']['eddsa'], endpoint)
    
    if 'errors' in result:
        print(f"GraphQL error creating material {name}: {result['errors']}")
        return None
    
    event_data = result.get('data', {}).get('createEconomicEvent', {}).get('economicEvent', {})
    resource = event_data.get('resourceInventoriedAs', {})
    
    if resource and resource.get('id'):
        material_resources[name] = {
            'id': resource.get('id'),
            'name': resource.get('name'),
            'spec_id': spec_id,
            'unit_id': unit_id  # Store unit for consumption
        }
        return material_resources[name]
    else:
        print(f"Failed to create material {name}: no resource returned")
        return None

# Create materials using specMaterial from GUI
if SPEC_MATERIAL_ID:
    print("Creating materials with GUI specMaterial...")
    
    # KROMA KRAFT Cardboard
    mat = create_material_resource(
        name="KROMA KRAFT Displaykarton (FSC)",
        note="PVC-free, FSC-certified cardboard for lamp structure",
        user_data=users_data['locilamp_manufacturer'],
        spec_id=SPEC_MATERIAL_ID,
        unit_id=units_data['mass']['id'],
        amount=0.3
    )
    if mat:
        print(f"  ✅ Created material: {mat['name']} (ID: {mat['id']})")
    
    # Acid-free paper
    mat = create_material_resource(
        name="Säurefreies Papier (FSC)",
        note="Transparent acid-free paper for lampshade",
        user_data=users_data['locilamp_manufacturer'],
        spec_id=SPEC_MATERIAL_ID,
        unit_id=units_data['mass']['id'],
        amount=0.02
    )
    if mat:
        print(f"  ✅ Created material: {mat['name']} (ID: {mat['id']})")
    
    # Textile cable - use piece unit since length not available
    mat = create_material_resource(
        name="Textilkabel H03VVH2-F",
        note="2-core textile cable, 2.4m length",
        user_data=users_data['locilamp_manufacturer'],
        spec_id=SPEC_MATERIAL_ID,
        unit_id=units_data['piece']['id'],
        amount=1
    )
    if mat:
        print(f"  ✅ Created material: {mat['name']} (ID: {mat['id']})")
    
    # E27 lamp socket
    mat = create_material_resource(
        name="E27 Lampenfassung",
        note="E27 socket with porcelain insulator",
        user_data=users_data['locilamp_manufacturer'],
        spec_id=SPEC_MATERIAL_ID,
        unit_id=units_data['piece']['id'],
        amount=1
    )
    if mat:
        print(f"  ✅ Created material: {mat['name']} (ID: {mat['id']})")
    
    # EU plug
    mat = create_material_resource(
        name="EU Stecker (2-PIN)",
        note="Type C European 2-pin plug",
        user_data=users_data['locilamp_manufacturer'],
        spec_id=SPEC_MATERIAL_ID,
        unit_id=units_data['piece']['id'],
        amount=1
    )
    if mat:
        print(f"  ✅ Created material: {mat['name']} (ID: {mat['id']})")
    
    # Switch
    mat = create_material_resource(
        name="Kippschalter",
        note="Toggle switch for cable",
        user_data=users_data['locilamp_manufacturer'],
        spec_id=SPEC_MATERIAL_ID,
        unit_id=units_data['piece']['id'],
        amount=1
    )
    if mat:
        print(f"  ✅ Created material: {mat['name']} (ID: {mat['id']})")
    
    print(f"\n✅ Created {len([m for m in material_resources.values() if m.get('id')])} material resources")
else:
    print("⚠️ SPEC_MATERIAL_ID not available - run instanceVariables cell first")

## Create Design Resource (using GUI specProjectDesign)

Create the LOCI LAMP design resource using the GUI's `specProjectDesign` specification.
This design will be CITED by the product during production (matching LinkDesignStep in the GUI).

In [ ]:
# ============================================================================
# CREATE DESIGN RESOURCE (using GUI specProjectDesign)
# ============================================================================
# This design resource uses the GUI's specProjectDesign specification.
# The product will CITE this design during production (LinkDesignStep).

design_resource = {}

def create_design_resource(name, description, repo_url, tags, user_data, spec_id, unit_id, 
                           licenses=None, process_id=None, endpoint=ENDPOINT):
    """Create a design resource using the GUI's specProjectDesign (matches CREATE_PROJECT mutation).
    
    Uses the library's send_signed for proper EdDSA authentication.
    """
    
    # Build metadata (matching GUI handleProjectCreation for designs)
    metadata = {
        "contributors": [],
        "licenses": licenses or [{"scope": "hardware", "licenseId": "CC-BY-SA-4.0"}],
        "relations": [],
        "declarations": {},  # Designs don't have declarations
        "remote": False,
        "design": False,  # This IS a design, not linking to one
        "machines": [],
        "materials": [],
        "productFilters": {}
    }
    
    ts = datetime.now(timezone.utc).isoformat()
    
    # If no process provided, we need to create one first
    if not process_id:
        # Create a simple process for design production
        process_vars = {
            "process": {
                "name": f"Design: {name}",
                "note": "Design resource creation process"
            }
        }
        process_query = """mutation($process:ProcessCreateParams!) {
            createProcess(process:$process) {
                process { id name }
            }
        }"""
        process_result = send_signed(process_query, process_vars, user_data['username'], 
                                     user_data['keyring']['eddsa'], endpoint)
        if 'errors' in process_result:
            print(f"Error creating process for design: {process_result['errors']}")
            return None
        process_id = process_result['data']['createProcess']['process']['id']
    
    # Build variables matching GUI CREATE_PROJECT mutation
    # Note: resourceMetadata expects a JSON string, so we serialize it
    variables = {
        "event": {
            "action": "produce",
            "provider": user_data['id'],
            "receiver": user_data['id'],
            "outputOf": process_id,
            "hasPointInTime": ts,
            "resourceClassifiedAs": tags,
            "resourceConformsTo": spec_id,
            "resourceQuantity": {
                "hasNumericalValue": 1,
                "hasUnit": unit_id
            },
            "toLocation": user_data.get('location_id'),
            "resourceMetadata": json.dumps(metadata)  # Serialize to JSON string
        },
        "newInventoriedResource": {
            "name": name,
            "note": description,
            "repo": repo_url,
            "license": "CC-BY-SA-4.0"
        }
    }
    
    query = """mutation($event:EconomicEventCreateParams!, $newInventoriedResource:EconomicResourceCreateParams) {
        createEconomicEvent(event:$event, newInventoriedResource:$newInventoriedResource) {
            economicEvent {
                id
                resourceInventoriedAs {
                    id
                    name
                    repo
                }
            }
        }
    }"""
    
    result = send_signed(query, variables, user_data['username'], user_data['keyring']['eddsa'], endpoint)
    
    if 'errors' in result:
        print(f"GraphQL error creating design: {result['errors']}")
        return None
    
    event_data = result.get('data', {}).get('createEconomicEvent', {}).get('economicEvent', {})
    resource = event_data.get('resourceInventoriedAs', {})
    
    if resource and resource.get('id'):
        return {
            'id': resource.get('id'),
            'name': resource.get('name'),
            'repo': resource.get('repo'),
            'spec_id': spec_id
        }
    else:
        print("Failed to create design: no resource returned")
        return None

# Create the LOCI LAMP design using specProjectDesign
if SPEC_PROJECT_DESIGN_ID:
    print("Creating LOCI LAMP design with GUI specProjectDesign...")
    
    design_resource = create_design_resource(
        name="LOCI LAMP V 2.0 Design",
        description="Open-source table lamp design. Self-assembly kit with sustainable materials. "
                    "Designed by LoCI LAMP Designer for Tchibo GmbH.",
        repo_url="https://github.com/locilamp/design-v2",
        tags=[
            "lamp",
            "table-lamp",
            "sustainable",
            "cardboard",
            "FSC",
            "DIY",
            "self-assembly",
            "open-source"
        ],
        user_data=users_data['locilamp_designer'],
        spec_id=SPEC_PROJECT_DESIGN_ID,
        unit_id=UNIT_ONE_ID or units_data['piece']['id'],
        licenses=[{"scope": "hardware", "licenseId": "CC-BY-SA-4.0"}]
    )
    
    if design_resource:
        print(f"✅ Created design resource:")
        print(f"   ID: {design_resource['id']}")
        print(f"   Name: {design_resource['name']}")
        print(f"   Repo: {design_resource['repo']}")
        DESIGN_RESOURCE_ID = design_resource['id']
    else:
        print("⚠️ Failed to create design resource")
        DESIGN_RESOURCE_ID = None
else:
    print("⚠️ specProjectDesign not available, skipping design creation")
    DESIGN_RESOURCE_ID = None

## Prepare and Upload Product Images

Create placeholder image files and upload them to the DPP service.

In [ ]:
# Create placeholder image files if they don't exist
ASSETS_DIR.mkdir(exist_ok=True, parents=True)

# Define image files for LOCI LAMP
image_files = {
    'locilamp_oliverschwartz_DSC05185.jpg': 'Main product image - LOCI LAMP overview',
    'locilamp_oliverschwartz_DSC09692.jpg': 'Detail image - Components and design',
    'locilamp_oliverschwartz_DSC09707.jpg': 'Detail image - Materials and assembly',
    'locilamp_oliverschwartz_DSC05218-landscape.jpg': 'Lifestyle image - LOCI LAMP in home setting',
}

# Create placeholder files (replace these with actual images)
for filename, description in image_files.items():
    filepath = ASSETS_DIR / filename
    if not filepath.exists():
        # Create a simple placeholder (in production, use actual image files)
        placeholder_content = f"[PLACEHOLDER] {description}\nProduct: LOCI LAMP V 2.0\nBrand: Tchibo GmbH\n"
        filepath.write_text(placeholder_content)
        print(f"✓ Created placeholder: {filename}")
    else:
        print(f"✓ Image exists: {filename}")

print(f"\n✓ Image files prepared in {ASSETS_DIR}")

## Upload Images to DPP Service

Upload the product images to the interfacer-dpp storage using signed requests.

In [ ]:
# Reload the if_gc1dpp module to pick up the fixed sign_message function
import importlib
import if_gc1dpp
importlib.reload(if_gc1dpp)
from if_gc1dpp import upload_file_on_dpp, calculate_file_checksum
print("✓ if_gc1dpp module reloaded with fixed sign_message function")

In [ ]:
# Upload images to DPP service using the Tchibo user credentials
# This mirrors the UploadFileOnDPP function from the GUI

user_for_upload = users_data['tchibo']
eddsa_public_key = user_for_upload['eddsa_public_key']
eddsa_private_key = user_for_upload['keyring']['eddsa']

print("Uploading images to DPP service...")
print(f"  Using credentials for: {user_for_upload['name']}")
print(f"  DPP endpoint: {DPP_URL}")
print()

for filename in image_files.keys():
    filepath = str(ASSETS_DIR / filename)
    try:
        # Call the actual upload function from if_gc1dpp
        attachment_response = upload_file_on_dpp(
            filepath,
            eddsa_public_key,
            eddsa_private_key,
            DPP_URL
        )
        images_data[filename] = attachment_response
        print(f"✓ Uploaded {filename}")
        print(f"    ID: {attachment_response.get('id', 'N/A')}")
        print(f"    URL: {attachment_response.get('url', 'N/A')}")
    except Exception as e:
        print(f"✗ Failed to upload {filename}: {e}")
        images_data[filename] = {'error': str(e), 'filename': filename}

# Save image upload data
with open(IMAGES_FILE, 'w') as f:
    json.dump(images_data, f, indent=2)
print(f"\n✓ Image data saved to {IMAGES_FILE}")

## Component Production

Produce the semi-finished components from raw materials.

In [ ]:
# Step 1: Produce cardboard lamelles and base from KROMA KRAFT
cur_res = action = event_note = amount = cur_pros = None
action = 'consume'
event_note = 'consume KROMA KRAFT cardboard for lamelles and base'
amount = 0.3
cur_pros = process_data['Create_locilamp_cardboard_components']
cur_res = res_data['kroma_kraft_cardboard']

event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id': cur_pros['id'], 'name': cur_pros['name']})

# Produce cardboard components
action = 'produce'
event_note = 'produce cardboard lamelles and core base'
amount = 1
res_data['locilamp_cardboard_components'] = {
    "res_ref_id": f'locilamp_cardboard_components-{random.randint(0, 10000)}',
    "name": 'LOCI LAMP cardboard lamelles and base',
    "spec_id": res_spec_data['locilamp_cardboard_components']['id']
}
cur_res = res_data['locilamp_cardboard_components']

event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Cardboard components produced (ID: {cur_res['id']})")

In [ ]:
# Step 2: Produce lampshade from acid-free paper
cur_res = action = event_note = amount = cur_pros = None
action = 'consume'
event_note = 'consume acid-free paper for lampshade'
amount = 0.02
cur_pros = process_data['Create_locilamp_lampshade']
cur_res = res_data['acidfree_paper']

event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id': cur_pros['id'], 'name': cur_pros['name']})

# Produce lampshade
action = 'produce'
event_note = 'produce transparent acid-free paper lampshade'
amount = 1
res_data['locilamp_lampshade'] = {
    "res_ref_id": f'locilamp_lampshade-{random.randint(0, 10000)}',
    "name": 'LOCI LAMP transparent lampshade',
    "spec_id": res_spec_data['locilamp_lampshade']['id']
}
cur_res = res_data['locilamp_lampshade']

event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Lampshade produced (ID: {cur_res['id']})")

In [ ]:
# Step 3: Assemble electrical components (cable, socket, plug, switch)
cur_res = action = event_note = amount = cur_pros = None
cur_pros = process_data['Create_locilamp_electrical']

# Consume textile cable
action = 'consume'
event_note = 'consume textile cable for electrical assembly'
amount = 1
cur_res = res_data['textile_cable']
event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

# Consume E27 socket
action = 'consume'
event_note = 'consume E27 socket for electrical assembly'
amount = 1
cur_res = res_data['e27_socket']
event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

# Consume EU plug
action = 'consume'
event_note = 'consume EU plug for electrical assembly'
amount = 1
cur_res = res_data['eu_plug']
event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

# Consume toggle switch
action = 'consume'
event_note = 'consume toggle switch for electrical assembly'
amount = 1
cur_res = res_data['toggle_switch']
event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id': cur_pros['id'], 'name': cur_pros['name']})

# Produce electrical assembly
action = 'produce'
event_note = 'produce electrical assembly with cable, socket, plug, and switch'
amount = 1
res_data['locilamp_electrical_assembly'] = {
    "res_ref_id": f'locilamp_electrical_assembly-{random.randint(0, 10000)}',
    "name": 'LOCI LAMP electrical assembly',
    "spec_id": res_spec_data['locilamp_electrical_assembly']['id']
}
cur_res = res_data['locilamp_electrical_assembly']

event_id, ts = create_event(users_data['locilamp_manufacturer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Electrical assembly produced (ID: {cur_res['id']})")

In [ ]:
# Step 4: Create the LOCI LAMP design
cur_res = action = event_note = amount = cur_pros = None
action = 'work'
event_note = 'design work for LOCI LAMP V 2.0'
cur_pros = process_data['Create_locilamp_design']
effort_spec = {}
effort_spec['unit_id'] = res_spec_data['design_work']['defaultUnit']
effort_spec['spec_id'] = res_spec_data['design_work']['id']
effort_spec['amount'] = 40  # 40 hours of design work

event_id, ts = create_event(users_data['locilamp_designer'], action, event_note, amount=0, process=cur_pros,
                 res_spec_data=res_spec_data, effort_spec=effort_spec, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'amount': effort_spec['amount']})
event_seq.append({'ts': ts, 'process_id': cur_pros['id'], 'name': cur_pros['name']})

# Produce design
action = 'produce'
event_note = 'produce LOCI LAMP V 2.0 design'
amount = 1
res_data['locilamp_design'] = {
    "res_ref_id": f'locilamp_design-{random.randint(0, 10000)}',
    "name": 'LOCI LAMP V 2.0 design',
    "spec_id": res_spec_data['locilamp_design']['id']
}
cur_res = res_data['locilamp_design']

event_id, ts = create_event(users_data['locilamp_designer'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ LOCI LAMP design created (ID: {cur_res['id']})")

## Transfer Components to Tchibo

Transfer all produced components to Tchibo for final assembly.

In [ ]:
# Transfer cardboard components to Tchibo
cur_res = res_data['locilamp_cardboard_components']
note = 'Transfer cardboard components from manufacturer to Tchibo'
action = 'transfer'
amount = 1
event_id, ts = make_transfer(users_data['locilamp_manufacturer'], action, note, users_data['tchibo'], amount, cur_res, locs_data, res_spec_data, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
print(f"✓ Cardboard components transferred to Tchibo")

# Transfer lampshade to Tchibo
cur_res = res_data['locilamp_lampshade']
note = 'Transfer lampshade from manufacturer to Tchibo'
action = 'transfer'
amount = 1
event_id, ts = make_transfer(users_data['locilamp_manufacturer'], action, note, users_data['tchibo'], amount, cur_res, locs_data, res_spec_data, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
print(f"✓ Lampshade transferred to Tchibo")

# Transfer electrical assembly to Tchibo
cur_res = res_data['locilamp_electrical_assembly']
note = 'Transfer electrical assembly from manufacturer to Tchibo'
action = 'transfer'
amount = 1
event_id, ts = make_transfer(users_data['locilamp_manufacturer'], action, note, users_data['tchibo'], amount, cur_res, locs_data, res_spec_data, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
print(f"✓ Electrical assembly transferred to Tchibo")

## Save All Data

In [ ]:
# ============================================================================
# SAVE ALL DATA FOR PRODUCTION NOTEBOOK
# ============================================================================

# Save all resources to file
with open(RES_FILE, 'w') as f:
    json.dump(res_data, f, indent=2)
print(f"✓ Resources saved to {RES_FILE}")

# Save process data
with open(PROCESS_FILE, 'w') as f:
    json.dump(process_data, f, indent=2)
print(f"✓ Process data saved to {PROCESS_FILE}")

# Save locations data (needed by Production for LocationStep)
LOCATIONS_FILE = get_filename('locations_data.json', ENDPOINT, USE_CASE)
with open(LOCATIONS_FILE, 'w') as f:
    json.dump(locs_data, f, indent=2)
print(f"✓ Locations saved to {LOCATIONS_FILE}")

# Save specs data (includes specDpp and locilamp_assembled)
with open(SPECS_FILE, 'w') as f:
    json.dump(res_spec_data, f, indent=2)
print(f"✓ Specs saved to {SPECS_FILE}")

# Save units data
with open(UNITS_FILE, 'w') as f:
    json.dump(units_data, f, indent=2)
print(f"✓ Units saved to {UNITS_FILE}")

# Save images data
with open(IMAGES_FILE, 'w') as f:
    json.dump(images_data, f, indent=2)
print(f"✓ Images saved to {IMAGES_FILE}")

# Store the parsed product data for the production notebook
PRODUCT_DATA_FILE = get_filename('product_data.json', ENDPOINT, USE_CASE)
with open(PRODUCT_DATA_FILE, 'w') as f:
    json.dump(loci_lamp_data, f, indent=2)
print(f"✓ Product data saved to {PRODUCT_DATA_FILE}")

# ============================================================================
# NEW: Save GUI-aligned resources (materials, design, specs)
# ============================================================================

# Save material resources (created with specMaterial)
MATERIALS_FILE = get_filename('material_resources.json', ENDPOINT, USE_CASE)
with open(MATERIALS_FILE, 'w') as f:
    json.dump(material_resources, f, indent=2)
print(f"✓ Material resources saved to {MATERIALS_FILE}")

# Save design resource (created with specProjectDesign)
DESIGN_FILE = get_filename('design_resource.json', ENDPOINT, USE_CASE)
with open(DESIGN_FILE, 'w') as f:
    json.dump(design_resource, f, indent=2)
print(f"✓ Design resource saved to {DESIGN_FILE}")

# Save GUI instanceVariables specs
GUI_SPECS_FILE = get_filename('gui_specs.json', ENDPOINT, USE_CASE)
gui_specs_data = {
    'specProjectDesign': {'id': SPEC_PROJECT_DESIGN_ID} if SPEC_PROJECT_DESIGN_ID else None,
    'specProjectProduct': {'id': SPEC_PROJECT_PRODUCT_ID} if SPEC_PROJECT_PRODUCT_ID else None,
    'specMaterial': {'id': SPEC_MATERIAL_ID} if SPEC_MATERIAL_ID else None,
    'specMachine': {'id': SPEC_MACHINE_ID} if SPEC_MACHINE_ID else None,
    'specDpp': {'id': SPEC_DPP_ID} if SPEC_DPP_ID else None,
    'unitOne': {'id': UNIT_ONE_ID} if UNIT_ONE_ID else None,
    'designResourceId': DESIGN_RESOURCE_ID
}
with open(GUI_SPECS_FILE, 'w') as f:
    json.dump(gui_specs_data, f, indent=2)
print(f"✓ GUI specs saved to {GUI_SPECS_FILE}")

print(f"\n📁 All data files saved for Production notebook.")

## Setup Complete - Summary

In [ ]:
print("\n" + "="*60)
print("LOCI LAMP SETUP COMPLETE")
print("="*60)
print(f"\n✓ {len(users_data)} users registered")
print(f"✓ {len(locs_data)} locations created")
print(f"✓ {len(units_data)} units registered")
print(f"✓ {len(res_spec_data)} resource specifications created")
print(f"✓ {len(process_data)} processes defined")
print(f"✓ {len(res_data)} resources created")
print(f"✓ {len(images_data)} images uploaded to DPP service")

print(f"\n📋 GUI instanceVariables specs (from server):")
print(f"  - specProjectDesign: {SPEC_PROJECT_DESIGN_ID or 'NOT AVAILABLE'}")
print(f"  - specProjectProduct: {SPEC_PROJECT_PRODUCT_ID or 'NOT AVAILABLE'}")
print(f"  - specMaterial: {SPEC_MATERIAL_ID or 'NOT AVAILABLE'}")
print(f"  - specMachine: {SPEC_MACHINE_ID or 'NOT AVAILABLE'}")
print(f"  - specDpp: {SPEC_DPP_ID or 'NOT AVAILABLE'}")

print(f"\n🎨 Design resource (for product to cite):")
print(f"  - ID: {DESIGN_RESOURCE_ID or 'NOT CREATED'}")
if design_resource:
    print(f"  - Name: {design_resource.get('name', 'N/A')}")
    print(f"  - Repo: {design_resource.get('repo', 'N/A')}")

print(f"\n🧱 Material resources (using specMaterial): {len(material_resources)}")
for name, mat in material_resources.items():
    print(f"  - {name}: {mat.get('id', 'N/A')}")

print(f"\n📦 Components produced (semi-finished):")
print(f"  - Cardboard lamelles and base (transferred to Tchibo)")
print(f"  - Transparent lampshade (transferred to Tchibo)")
print(f"  - Electrical assembly (transferred to Tchibo)")

print(f"\n📄 Product info from CSV:")
po = loci_lamp_data.get('Product Overview', {})
print(f"  - Brand: {po.get('Brand Name', 'N/A')}")
print(f"  - Product: {po.get('Product Name', 'N/A')}")
print(f"  - Model: {po.get('Model Name', 'N/A')}")
print(f"  - Country: {po.get('Country of Origin', 'N/A')}")

print(f"\n✅ You can now run the Production notebook to create the LOCI LAMP with DPP.")